# pop Phase 2 -- one-notebook Colab run (resumable)

Runs **all** of Phase 2 (tokenizer -> 10-epoch pretrain -> 6 finetuned systems -> generate -> eval)
via `scripts/run_training.py`, with every artifact stored on **your Google Drive** so nothing is
lost when a Colab session ends.

**Before first run** (see `docs/colab-runbook.md` for details): create the Drive folder
`MyDrive/pop_phase2/` and upload `pop_repo.zip` (built locally with `git archive`) into it.

**Every session**: Runtime > Change runtime type > pick a GPU (**A100** with Colab Pro,
~3-5 h total; **T4** on the free tier, ~35-50 h across sessions -- the code auto-scales its
batching to the GPU, effective batch stays 64), then Runtime > **Run all**.
Safe to re-run any time -- completed steps are skipped, an interrupted training step resumes
from its latest epoch checkpoint, and progress is always visible in
`Drive/pop_phase2/logs/train/phase2/STATUS.md`. One rule: don't switch GPU *class* while a
training step is mid-run (between steps is fine; see the runbook's "Which GPU?" section).

In [ ]:
import sys

print("Python", sys.version)
assert sys.version_info >= (3, 11), "pop needs Python >= 3.11; this Colab runtime is older"
!nvidia-smi

In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/pop_phase2")
for sub in ("outputs", "results", "logs"):
    (BASE / sub).mkdir(parents=True, exist_ok=True)

ZIP = BASE / "pop_repo.zip"
assert ZIP.is_file(), (
    f"Upload pop_repo.zip to {ZIP.parent} first -- build it locally with\n"
    "  git archive --format=zip -o dist/pop_repo.zip HEAD\n"
    "then drag dist/pop_repo.zip into Drive/pop_phase2/ (see docs/colab-runbook.md)."
)
print("Drive workspace ready:", BASE)

In [ ]:
%cd /content
!rm -rf /content/repo
!unzip -q /content/drive/MyDrive/pop_phase2/pop_repo.zip -d /content/repo
%cd /content/repo
%pip install -q -e .

In [ ]:
# Point the repo's outputs/, results/, logs/ at Drive so checkpoints, metrics, and
# progress logs all survive session resets. The repo ships a committed results/
# directory; its contents are copied onto Drive once before the swap.
import os
import shutil
from pathlib import Path

BASE = Path("/content/drive/MyDrive/pop_phase2")
REPO = Path("/content/repo")
for sub in ("outputs", "results", "logs"):
    drive_dir = BASE / sub
    repo_dir = REPO / sub
    if repo_dir.is_symlink():
        repo_dir.unlink()
    elif repo_dir.exists():
        shutil.copytree(repo_dir, drive_dir, dirs_exist_ok=True)
        shutil.rmtree(repo_dir)
    os.symlink(drive_dir, repo_dir)
print("outputs/, results/, logs/ now live on Drive:", BASE)

### Optional: Weights & Biases

Training runs fine without W&B (file logs + STATUS.md cover tracking). If you want W&B
dashboards too, add a new cell with `import wandb; wandb.login()` and paste **your own** key
when prompted -- the key is never stored in this notebook or the repo. It is deliberately not a
default cell so that **Run all** never stalls waiting for input.

In [ ]:
# The long-running cell. Re-running after any interruption continues where it left off.
# If it halts with "EM GATE", see docs/colab-runbook.md -- that is a deliberate stop.
!python scripts/run_training.py

In [ ]:
import json
from pathlib import Path

status = Path("logs/train/phase2/STATUS.md")
if status.exists():
    print(status.read_text(encoding="utf-8"))
summary = Path("logs/train/phase2/SUMMARY.md")
if summary.exists():
    print(summary.read_text(encoding="utf-8"))
for path in sorted(Path("results").glob("finetune_*_test.json")):
    metrics = json.loads(path.read_text(encoding="utf-8"))["metrics"]
    print(
        f"{path.name}: EM={metrics['em']:.4f} CodeBLEU={metrics['codebleu']:.4f} "
        f"syntax={metrics['syntax_valid_rate']:.4f} n={metrics['n']}"
    )

### If the session disconnects or hits the GPU quota

Normal and expected on the free tier. Reopen this notebook and **Run all** again: finished
steps are skipped instantly, and a training step that was mid-run resumes from its last
completed epoch. The full matrix takes multiple sessions across a few days of free-tier quota;
the headline A-vs-B systems (`A_ep10`, `B_seed0`) finish first on purpose. Progress at any
time: `Drive/pop_phase2/logs/train/phase2/STATUS.md`.